In [11]:
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.coordinates import EarthLocation, SkyCoord
from astropy.time import Time
from astroplan import Observer, FixedTarget
from pytz import timezone

In [4]:
pip install astroplan

Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for astroplan: filename=astroplan-0.10.1-py3-none-any.whl size=83919 sha256=252560a4d8705a44745ebbf7b1b36f9d7a41e4e9004b928d698cb793a10c70ac
  Stored in directory: c:\users\golod\appdata\local\pip\cache\wheels\10\8d\c2\c5abd8ed9f9ac3f737a80116bae9ae1c64b2b1ef1a5cc39ba8
Successfully built astroplan
Note: you may need to restart the kernel to use updated packages.


In [7]:
class Observatory:
    def __init__(self, observer_obj):
        """Initializes with a pre-built astroplan Observer object."""
        self.observer = observer_obj

    @classmethod
    def from_known_site(cls, site_name):
        """Creates an observatory from astroplan's built-in database."""
        # e.g., Observer.at_site('subaru')
        return cls(Observer.at_site(site_name))

    @classmethod
    def from_custom_location(cls, name, lon_str, lat_str, elevation_m, tz_str):
        """Creates a custom observatory using string coordinates from the docs."""
        location = EarthLocation.from_geodetic(lon_str, lat_str, elevation_m * u.m)
        observer = Observer(name=name,
                            location=location,
                            timezone=timezone(tz_str))
        return cls(observer)

    def can_see(self, target, time, min_alt=30*u.deg):
        """Checks if the target is visible at a specific global time."""
        is_night = self.observer.is_night(time, horizon=-18*u.deg)
        is_up = self.observer.target_is_up(time, target.astroplan_target, horizon=min_alt)
        return is_night and is_up


class Target:
    def __init__(self, fixed_target_obj):
        """Initializes with a pre-built astroplan FixedTarget object."""
        self.astroplan_target = fixed_target_obj

    @classmethod
    def from_coordinates(cls, name, ra_str, dec_str):
        """Creates a target using string coordinates (e.g., '19h50m47.6s')."""
        coord = SkyCoord(ra_str, dec_str, frame='icrs')
        return cls(FixedTarget(name=name, coord=coord))

    @classmethod
    def from_name(cls, name):
        """Retrieves coordinates automatically using the CDS name resolver."""
        return cls(FixedTarget.from_name(name))
    
    @classmethod
    def generate_random(cls, name="Random_Transient"):
        """Generates evenly distributed coordinates on the sphere."""
        ra = np.random.uniform(0, 360)
        dec = np.degrees(np.arcsin(np.random.uniform(-1, 1)))
        coord = SkyCoord(ra=ra*u.deg, dec=dec*u.deg)
        return cls(FixedTarget(coord=coord, name=name))

In [15]:
# --- Re-initialize the Observatories to get the new methods ---

# MAST/DeepSpec
mast = Observatory.from_custom_location(
    name="MAST", lon_str="35d02m00s", lat_str="+30d03m00s", elevation_m=400, tz_str="Asia/Jerusalem"
)

# NTT/SOXS
soxs = Observatory.from_custom_location(
    name="SOXS", lon_str="-70d44m00s", lat_str="-29d15m00s", elevation_m=2400, tz_str="America/Santiago"
)

# LDT/WILDS
wilds = Observatory.from_custom_location(
    name="WILDS", lon_str="-111d44m25s", lat_str="+34d44m40s", elevation_m=2360, tz_str="America/Phoenix"
)

In [9]:
# --- 2. Generate 1000 Targets & Random Times ---

# Set start date to today
start_time = Time('2026-03-04 11:40:00')

# Create an array of 1000 random targets
print("Generating 1000 targets and times...")
targets = [Target.generate_random(name=f"T_{i}") for i in range(1000)]

# Add a random number of days (between 0 and 365) to the start time for each target
random_days = np.random.uniform(0, 365, 1000)
times = start_time + random_days * u.day

Generating 1000 targets and times...


In [12]:
# --- 3. Run the Simulation ---

results = []

for i in range(1000):
    targ = targets[i]
    t = times[i]
    
    # Check visibility for all three telescopes at this specific time
    results.append({
        "Target_ID": targ.astroplan_target.name,
        "RA (deg)": round(targ.astroplan_target.coord.ra.deg, 2),
        "Dec (deg)": round(targ.astroplan_target.coord.dec.deg, 2),
        "Time (UTC)": t.iso[:19], # Truncating to seconds for clean formatting
        "MAST_Visible": mast.can_see(targ, t),
        "SOXS_Visible": soxs.can_see(targ, t),
        "WILDS_Visible": wilds.can_see(targ, t)
    })

# --- 4. Display the Results ---

# Convert the list of dictionaries into a Pandas DataFrame for easy viewing
df = pd.DataFrame(results)

print("\n--- Simulation Complete ---\n")
print(df.head(10)) # Print the first 10 rows

print("\n--- Total Detections over 1 Year ---")
print(f"MAST (Israel):  {df['MAST_Visible'].sum()} targets")
print(f"SOXS (Chile):   {df['SOXS_Visible'].sum()} targets")
print(f"WILDS (USA):    {df['WILDS_Visible'].sum()} targets")


--- Simulation Complete ---

  Target_ID  RA (deg)  Dec (deg)           Time (UTC)  MAST_Visible  \
0       T_0    327.05      -9.39  2026-09-09 14:25:14         False   
1       T_1     44.39     -40.65  2026-05-28 12:16:13         False   
2       T_2    246.60     -54.15  2026-08-04 16:52:55         False   
3       T_3    276.43       1.72  2026-12-28 22:22:12         False   
4       T_4    338.86     -23.33  2026-07-27 21:11:35         False   
5       T_5     90.50      18.74  2026-03-20 15:42:44         False   
6       T_6    160.36       5.48  2026-07-10 14:54:36         False   
7       T_7    176.78      29.40  2026-11-17 10:55:54         False   
8       T_8    183.80      46.45  2026-04-22 10:16:45         False   
9       T_9     18.54      48.29  2026-08-11 12:17:56         False   

   SOXS_Visible  WILDS_Visible  
0         False          False  
1         False          False  
2         False          False  
3         False          False  
4         False        

In [13]:
class Observatory:
    def __init__(self, observer_obj):
        """Initializes with a pre-built astroplan Observer object."""
        self.observer = observer_obj

    @classmethod
    def from_known_site(cls, site_name):
        """Creates an observatory from astroplan's built-in database."""
        return cls(Observer.at_site(site_name))

    @classmethod
    def from_custom_location(cls, name, lon_str, lat_str, elevation_m, tz_str):
        """Creates a custom observatory using string coordinates from the docs."""
        location = EarthLocation.from_geodetic(lon_str, lat_str, elevation_m * u.m)
        observer = Observer(name=name, location=location, timezone=timezone(tz_str))
        return cls(observer)

    def can_see(self, target, time, min_alt=30*u.deg):
        """Checks if the target is visible at a specific global time."""
        is_night = self.observer.is_night(time, horizon=-18*u.deg)
        is_up = self.observer.target_is_up(time, target.astroplan_target, horizon=min_alt)
        return is_night and is_up

    def get_snr_parameters(self, target, time):
        """Extracts the physical parameters needed for the radiometric SNR equation."""
        
        # 1. Target Position and Airmass
        target_altaz = self.observer.altaz(time, target.astroplan_target)
        airmass = target_altaz.secz.value # Extracts the float value of sec(zenith)
        
        # 2. Moon Parameters (Noise Contributors)
        moon_illum = self.observer.moon_illumination(time)
        moon_pos = self.observer.moon_altaz(time)
        moon_sep = moon_pos.separation(target_altaz).deg # Distance from target
        moon_alt = moon_pos.alt.deg # Is the moon even above the horizon?
        
        # 3. Sun Parameters (Twilight Contributors)
        sun_alt = self.observer.sun_altaz(time).alt.deg
        
        return {
            "airmass": round(airmass, 3),
            "moon_illumination": round(moon_illum, 3), # 0.0 (New) to 1.0 (Full)
            "moon_separation_deg": round(moon_sep, 2),
            "moon_altitude_deg": round(moon_alt, 2),
            "sun_altitude_deg": round(sun_alt, 2)
        }

In [16]:
# --- 3. Run the Simulation ---

results = []

print("Running visibility and SNR parameter checks (this may take a minute)...")

for i in range(1000):
    targ = targets[i]
    t = times[i]
    
    # 1. Check basic visibility first
    mast_vis = mast.can_see(targ, t)
    soxs_vis = soxs.can_see(targ, t)
    wilds_vis = wilds.can_see(targ, t)
    
    # 2. Set up the base row data
    row_data = {
        "Target_ID": targ.astroplan_target.name,
        "RA (deg)": round(targ.astroplan_target.coord.ra.deg, 2),
        "Dec (deg)": round(targ.astroplan_target.coord.dec.deg, 2),
        "Time (UTC)": t.iso[:19],
        "MAST_Visible": mast_vis,
        "SOXS_Visible": soxs_vis,
        "WILDS_Visible": wilds_vis
    }
    
    # 3. Add the exact SNR parameters ONLY if the telescope can see the target
    if mast_vis:
        params = mast.get_snr_parameters(targ, t)
        for key, value in params.items(): 
            row_data[f"MAST_{key}"] = value
            
    if soxs_vis:
        params = soxs.get_snr_parameters(targ, t)
        for key, value in params.items(): 
            row_data[f"SOXS_{key}"] = value
            
    if wilds_vis:
        params = wilds.get_snr_parameters(targ, t)
        for key, value in params.items(): 
            row_data[f"WILDS_{key}"] = value

    results.append(row_data)

# --- 4. Display the Results ---

# Convert the list of dictionaries into a Pandas DataFrame
df = pd.DataFrame(results)

print("\n--- Simulation Complete ---\n")

# Display a subset of columns for the first few successful MAST detections
mast_successes = df[df['MAST_Visible'] == True]
if not mast_successes.empty:
    print("Example of successful MAST detections with SNR parameters:")
    print(mast_successes[['Target_ID', 'Time (UTC)', 'MAST_airmass', 'MAST_moon_illumination', 'MAST_sun_altitude_deg']].head())
else:
    print("No MAST targets were visible in this run.")

print("\n--- Total Detections over 1 Year ---")
print(f"MAST (Israel):  {df['MAST_Visible'].sum()} targets")
print(f"SOXS (Chile):   {df['SOXS_Visible'].sum()} targets")
print(f"WILDS (USA):    {df['WILDS_Visible'].sum()} targets")

Running visibility and SNR parameter checks (this may take a minute)...

--- Simulation Complete ---

Example of successful MAST detections with SNR parameters:
   Target_ID           Time (UTC)  MAST_airmass  MAST_moon_illumination  \
11      T_11  2026-05-16 20:06:37         1.070                   0.002   
31      T_31  2026-03-16 19:07:57         1.075                   0.061   
44      T_44  2026-06-19 19:10:18         1.901                   0.282   
60      T_60  2026-07-14 22:04:59         1.309                   0.005   
67      T_67  2026-12-29 02:02:45         1.431                   0.681   

    MAST_sun_altitude_deg  
11                 -36.19  
31                 -42.46  
44                 -25.40  
60                 -38.18  
67                 -32.02  

--- Total Detections over 1 Year ---
MAST (Israel):  95 targets
SOXS (Chile):   102 targets
WILDS (USA):    88 targets
